# Nested tip racks

A nested tip rack (NTR) is a tip rack that stacks on top of another rack of the same type. A stack
of nested racks takes the deck space of one rack. When the top rack is used up, it is removed, and
the rack below it becomes reachable.

In PyLabRobot, a nested tip rack is a {class}`~pylabrobot.resources.tip_rack.StandingTipRack` with
a `stacking_z_height`: the height, in mm, that one rack adds to a stack of identical racks. A
standing rack stands on its own frame. An {class}`~pylabrobot.resources.tip_rack.EmbeddedTipRack`
sits down in a carrier site and does not nest.

```{note}
`NestedTipRack` is deprecated. Use
{class}`~pylabrobot.resources.tip_rack.StandingTipRack` with a `stacking_z_height` instead.
```

## A predefined nested tip rack

The resource library has nested tip racks, for example the Hamilton 10, 50 and 300 uL NTRs. Each
one is a `StandingTipRack`:

In [ ]:
from pylabrobot.resources import StandingTipRack, hamilton_96_tiprack_50uL_NTR

rack = hamilton_96_tiprack_50uL_NTR(name="ntr")

assert isinstance(rack, StandingTipRack)
assert rack.get_size_z() == 55.0
assert rack.stacking_z_height == 16.0

## Stacking nested tip racks

Put nested racks in a {class}`~pylabrobot.resources.ResourceStack` with
`direction="z"`. A standing rack that has a `stacking_z_height` and is placed on another such rack
nests into it: it stands `stacking_z_height` above the rack below, not `size_z` above it. A stack
of `N` identical racks is `size_z + (N - 1) * stacking_z_height` tall.

In [ ]:
from pylabrobot.resources import ResourceStack

stack = ResourceStack(name="ntr_stack", direction="z")
racks = [hamilton_96_tiprack_50uL_NTR(name=f"ntr_{i}") for i in range(3)]
for r in racks:
  stack.assign_child_resource(r)

assert [r.location.z for r in racks] == [0, 16.0, 32.0]
assert stack.get_size_z() == 55.0 + 2 * 16.0

Only the top rack of a stack is available for tip handling. A rack with another rack on top of it,
or with a lid on it, is not. The Hamilton STAR and Prep drivers and the legacy `LiquidHandler`
check `available_for_tip_handling`, and refuse to pick up tips from such a rack or to drop tips
into it. Remove the top rack to make the rack below it available:

In [ ]:
assert [r.available_for_tip_handling for r in racks] == [False, False, True]

stack.unassign_child_resource(racks[-1])

assert racks[1].available_for_tip_handling
assert stack.get_size_z() == 55.0 + 16.0

A rack does not nest into a rack that has a lid on it. The upper rack stands on the lid instead.

## Nested tip racks on a carrier

A carrier for nested tip racks holds a z `ResourceStack` on each site. For example, each site of
`hamilton_tip_carrier_L5_ntr_a00` (Hamilton `TIP_CAR_NTR_A00`) holds a stack. Assigning a rack to
a site puts the rack on top of the stack on that site:

In [ ]:
from pylabrobot.resources import hamilton_tip_carrier_L5_ntr_a00

carrier = hamilton_tip_carrier_L5_ntr_a00(name="ntr_carrier")
carrier[0] = hamilton_96_tiprack_50uL_NTR(name="lower")
carrier[0] = hamilton_96_tiprack_50uL_NTR(name="upper")

site_stack = carrier[0].resource
assert isinstance(site_stack, ResourceStack)
assert [r.name for r in site_stack.children] == ["lower", "upper"]
assert carrier.get_resource("upper").location.z == 16.0

## Defining a nested tip rack

To define a nested tip rack, create a `StandingTipRack` and give it a `stacking_z_height`. To
measure it, stack two racks and measure the height of the pair:

```
stacking_z_height = height_of_two_stacked_racks - size_z
```

The other arguments are the same as for any tip rack (see the {doc}`TipSpot guide <tip-spot>`).
`frame_height` is `0` for a rack that holds its tips on its top face:

In [ ]:
from pylabrobot.resources import TipSpot, create_ordered_items_2d, hamilton_tip_50uL

custom_rack = StandingTipRack(
  name="custom_ntr",
  size_x=127.76,
  size_y=85.48,
  size_z=55.0,
  stacking_z_height=16.0,
  ordered_items=create_ordered_items_2d(
    TipSpot,
    num_items_x=12,
    num_items_y=8,
    dx=10.78,
    dy=7.64,
    dz=55.0,
    item_dx=9.0,
    item_dy=9.0,
    size_x=7.2,
    size_y=7.2,
    make_tip=hamilton_tip_50uL,
    name_prefix="custom_ntr",
  ),
  frame_height=0.0,
)

assert custom_rack.num_items == 96
assert custom_rack.stacking_z_height == 16.0

`stacking_z_height` is part of the serialized rack, so a deserialized rack nests the same way:

In [ ]:
from pylabrobot.resources import Resource

loaded_rack = Resource.deserialize(custom_rack.serialize())

assert isinstance(loaded_rack, StandingTipRack)
assert loaded_rack.stacking_z_height == 16.0

See {class}`~pylabrobot.resources.tip_rack.StandingTipRack` and
{class}`~pylabrobot.resources.ResourceStack` for the API reference.